# Web-ASR — chạy Gradio trên một Colab
Backend và giao diện chạy trong **cùng một process**, khởi động bằng notebook này. Mic/loa nằm trong browser của bạn.

1. Chọn **Runtime → Change runtime type → T4 GPU** khi chạy thật.
2. Có repo đầy đủ gồm `Web-ASR` và `Sub-ASR`, hoặc nhập URL Git ở cell đầu.
3. Điền key trong `.env` gốc repo hoặc Colab Secrets. Không chia sẻ notebook có secret trong source/output.
4. Mở URL Gradio trong tab riêng, đăng nhập, cấp quyền mic; dùng tai nghe. Link chỉ tồn tại khi runtime đang chạy.

`FAKE=True` chỉ kiểm tra đường web, không nhận dạng tiếng nói thật. Colab là môi trường thử nghiệm, không phải dịch vụ luôn hoạt động. Kiểm tra chính sách Colab của tài khoản trước khi dùng tính năng share.

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, time, re, getpass, signal

FAKE = False
SHARE = True
REPO_URL = ''  # URL Git của repo đầy đủ; để trống nếu đã có source
REPO_BRANCH = ''
REPO_DIR = Path('/content/ASR') if Path('/content').exists() else Path.cwd()
if not (REPO_DIR / 'Web-ASR/pyproject.toml').exists():
    candidates = [Path.cwd(), Path.cwd().parent]
    found = next((p for p in candidates if (p/'Web-ASR/pyproject.toml').exists()), None)
    if found: REPO_DIR = found
    elif REPO_URL:
        command = ['git', 'clone'] + (['--branch', REPO_BRANCH] if REPO_BRANCH else []) + [REPO_URL, str(REPO_DIR)]
        subprocess.run(command, check=True)
    else: raise RuntimeError('Upload/clone repo đầy đủ hoặc điền REPO_URL và chạy lại.')
REPO_DIR = REPO_DIR.resolve()
PROJECT = REPO_DIR / 'Web-ASR'
assert (REPO_DIR/'Sub-ASR/voice.py').exists(), 'Thiếu Sub-ASR'
print('Project:', PROJECT)


## Cấu hình server
Notebook tự tìm `.env` trong repo hoặc file upload ở `/content/.env`, đọc trước khi hỏi thông tin thiếu. `ASR_ENV_PATH` chỉ định file khác. Thứ tự ưu tiên: `.env` < environment < Colab Secrets. Chỉ in đường dẫn file, không in secret. Username/password bảo vệ link web, khác với API key LLM.


In [ ]:
# Tìm file upload ở /content/.env hoặc .env trong repo; có thể chỉ định ASR_ENV_PATH.
explicit_env = os.environ.get('ASR_ENV_PATH')
candidates = [REPO_DIR / '.env', REPO_DIR.parent / '.env', Path.cwd() / '.env']
ENV_FILE = Path(explicit_env).expanduser() if explicit_env else next((p for p in candidates if p.is_file()), candidates[0])
ENV_FILE = ENV_FILE.resolve()
if explicit_env and not ENV_FILE.is_file():
    raise FileNotFoundError(f'Không tìm thấy ASR_ENV_PATH: {ENV_FILE}')
try:
    from dotenv import dotenv_values
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv>=1,<2'], check=True)
    from dotenv import dotenv_values
# Thứ tự ưu tiên: .env < biến môi trường < Colab Secrets.
SERVER_ENV = {k: v for k, v in dotenv_values(ENV_FILE).items() if v is not None} if ENV_FILE.is_file() else {}
SERVER_ENV.update(os.environ)
SERVER_ENV['ASR_ENV_PATH'] = str(ENV_FILE)
try:
    from google.colab import userdata
except ImportError:
    userdata = None
for name in ['LLM_API_KEY', 'LLM_BASE_URL', 'LLM_MODEL', 'TURN_MODEL', 'WEB_USERNAME', 'WEB_PASSWORD']:
    if userdata is not None:
        try: value = userdata.get(name)
        except Exception: value = None
        if value: SERVER_ENV[name] = value
SERVER_ENV.update(WEB_FAKE=str(FAKE).lower(), WEB_SHARE=str(SHARE).lower(), WEB_HOST='0.0.0.0' if SHARE else '127.0.0.1')
if SHARE:
    SERVER_ENV['WEB_USERNAME'] = SERVER_ENV.get('WEB_USERNAME') or 'demo'
    if not SERVER_ENV.get('WEB_PASSWORD'):
        SERVER_ENV['WEB_PASSWORD'] = getpass.getpass('Mật khẩu đăng nhập web: ')
    if not SERVER_ENV['WEB_PASSWORD']: raise ValueError('Mật khẩu không được trống')
if not FAKE:
    if not SERVER_ENV.get('LLM_BASE_URL'): SERVER_ENV['LLM_BASE_URL'] = input('LLM base URL: ').strip()
    if not SERVER_ENV.get('LLM_MODEL'): SERVER_ENV['LLM_MODEL'] = input('LLM model: ').strip()
    if not SERVER_ENV.get('LLM_API_KEY'): SERVER_ENV['LLM_API_KEY'] = getpass.getpass('LLM API key: ')
print('File cấu hình:', ENV_FILE if ENV_FILE.is_file() else 'chưa có .env; dùng environment/Secrets/nhập tay')
print('Đã chuẩn bị cấu hình; secret không được in ra.')


## Tạo môi trường uv và chuẩn bị model
Python 3.12 trong `.venv` độc lập với Python kernel Colab. Lần đầu tải CUDA/model có thể mất vài phút và dung lượng ổ đĩa đáng kể. Chạy fake bỏ qua GPU/model/TTS hệ thống.

In [ ]:
if not shutil.which('uv'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
UV = shutil.which('uv')
subprocess.run([UV, 'python', 'install', '3.12'], check=True)
EXTRAS = [] if FAKE else ['--extra', 'cuda']
if not FAKE:
    if not shutil.which('nvidia-smi'): raise RuntimeError('Chưa có GPU: chọn T4 GPU rồi kết nối lại runtime.')
    subprocess.run(['nvidia-smi'], check=True)
    if not shutil.which('espeak-ng'):
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'espeak-ng'], check=True)
subprocess.run([UV, 'sync', '--locked', *EXTRAS], cwd=PROJECT, check=True)
if not FAKE:
    setup_env = dict(SERVER_ENV, PYTHONPATH=str(REPO_DIR/'Sub-ASR'))
    subprocess.run([UV, 'run', '--locked', *EXTRAS, 'python', '-m', 'sub_asr.setup', '--config', str(REPO_DIR/'Sub-ASR/config.toml')], cwd=PROJECT, env=setup_env, check=True)
print('Môi trường sẵn sàng.')


## Khởi chạy / khởi chạy lại
Cell này dừng process của lần chạy trước trong cùng kernel rồi mở đúng một server. Model được nạp một lần ở server. Không đóng cell kết nối Colab khi đang sử dụng.

In [ ]:
def stop_server():
    global WEB_PROCESS, WEB_LOG
    process = globals().get('WEB_PROCESS')
    if process is not None and process.poll() is None:
        process.send_signal(signal.SIGINT)
        try: process.wait(timeout=12)
        except subprocess.TimeoutExpired:
            process.terminate()
            try: process.wait(timeout=5)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=5)
    log = globals().get('WEB_LOG')
    if log is not None: log.close()
stop_server()
LOG_PATH = PROJECT / 'server.log'
WEB_LOG = LOG_PATH.open('w')
WEB_PROCESS = subprocess.Popen([UV, 'run', '--locked', *EXTRAS, 'python', '-u', 'app.py'], cwd=PROJECT, env=SERVER_ENV, stdout=WEB_LOG, stderr=subprocess.STDOUT)
print('Đã khởi chạy PID', WEB_PROCESS.pid, '— theo dõi URL ở cell tiếp theo.')


## Xem URL và trạng thái
Chạy lại cell này khi model tải lâu. Nếu server dừng, xem lỗi ở log rồi sửa cấu hình. Đăng nhập bằng WEB_USERNAME / mật khẩu vừa đặt.

In [ ]:
from IPython.display import display, Markdown

for _ in range(30):  # Đợi tối đa ~60 giây
    time.sleep(2)
    content = LOG_PATH.read_text(errors='replace')
    urls = re.findall(r'https://[^\s]+\.gradio\.live', content)
    if urls or WEB_PROCESS.poll() is not None:
        break

print(content[-10000:])
print('Process:', 'đang chạy' if WEB_PROCESS.poll() is None else f'đã dừng, exit={WEB_PROCESS.returncode}')
if urls:
    display(Markdown(f'### 👉 [Mở Web-ASR]({urls[-1]})'))


## Dừng server khi sử dụng xong
Chỉ chạy cell dưới khi muốn dừng web. Chạy lại cell khởi động để mở lại. Session/history hiện ở RAM, dừng process sẽ xóa phiên.

In [ ]:
stop_server()
print('Đã dừng server.')
